# Simulation code for section 4

In [2]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# =========================================================
# Paths for notebook mode
# Current notebook location:
#   /MFDNN_code/Simulation
# =========================================================
SIMULATION_DIR = Path.cwd().resolve()
PROJECT_ROOT = SIMULATION_DIR.parent
METHOD_DIR = PROJECT_ROOT / "Method"
DATA_DIR = SIMULATION_DIR / "Data"
RESULTS_DIR = SIMULATION_DIR / "Results"

sys.path.insert(0, str(PROJECT_ROOT))

from Method.mfdnn import MFDNN, MFDNN_predict

torch.manual_seed(756)
np.random.seed(756)

out_path = RESULTS_DIR / "Simulation_MFDNN.csv"

configurations = [
    {"T": 16, "n": 200},
    {"T": 16, "n": 400},
    {"T": 32, "n": 200},
    {"T": 32, "n": 400},
]

lam1_values = [0.5, 1, 1.5, 2, 2.5, 3]
lam2_values = [0, 0.001, 0.01, 0.1, 0.5, 1]

model_params = {
    "num_basis": [5, 5],
    "layer_sizes": [64, 64],
    "epochs": 100,
    "val_ratio": 0.25,
    "patience": 10,
    "basis_type": "bspline",
    "degree": 3,
}

ground_truth = {
    0: {0, 1},
    1: {1, 4, 5},
    2: {0, 2, 3, 5},
    3: {0, 1},
    4: {1, 4, 5},
    5: {0, 2, 3, 5},
}

epsilon = 0.01

# For a quick test, we set frun = 5 here.
# To fully reproduce the simulation results reported in the paper, please change frun to 50.
frun = 50


def calculate_selection_metrics(l21_norms, true_vars, epsilon=0.01):
    selected_vars = set(i for i, norm in enumerate(l21_norms) if norm > epsilon)

    tp = len(selected_vars & true_vars)
    fp = len(selected_vars - true_vars)
    fn = len(true_vars - selected_vars)

    denom = 2 * tp + fp + fn
    f1_score = (2 * tp) / denom if denom > 0 else 0.0
    perfect_selection = 1.0 if selected_vars == true_vars else 0.0

    return f1_score, perfect_selection, selected_vars


def select_best_hyperparameters(
    X_train,
    y_train,
    true_vars,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
    epsilon=0.01,
):
    selection_info = {}

    y_train_mean = np.mean(y_train)
    y_train_std = np.std(y_train)

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, l21 = MFDNN(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                )

                current_mse = (
                    min(val_losses)
                    if len(val_losses) > 0
                    else np.mean(train_losses[-10:])
                )
                f1_score, perfect_selection, selected_vars = calculate_selection_metrics(
                    l21, true_vars, epsilon
                )

                selection_info[(i, j)] = {
                    "model": model,
                    "lam1": lam1,
                    "lam2": lam2,
                    "f1_score": f1_score,
                    "mse": current_mse,
                    "selected_vars": list(selected_vars),
                    "y_mean": y_train_mean,
                    "y_std": y_train_std,
                    "perfect_selection": perfect_selection,
                }
            except Exception:
                continue

    if not selection_info:
        return {
            "model": None,
            "lam1": 1,
            "lam2": 0.1,
            "f1_score": 0.0,
            "mse": np.inf,
            "selected_vars": [],
            "y_mean": y_train_mean,
            "y_std": y_train_std,
            "perfect_selection": 0.0,
        }

    best_key = max(
        selection_info,
        key=lambda k: (selection_info[k]["f1_score"], -selection_info[k]["mse"]),
    )
    return selection_info[best_key]


def evaluate_on_test_set(best_candidate, X_test, y_test, p, domain_range, model_params):
    if best_candidate["model"] is None:
        return np.inf, best_candidate["f1_score"]

    try:
        y_mean = best_candidate["y_mean"]
        y_std = best_candidate["y_std"]

        test_predictions_normalized = MFDNN_predict(
            p,
            best_candidate["model"],
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        )

        test_predictions_original = (
            test_predictions_normalized.detach().numpy() * y_std + y_mean
        )
        test_mse = np.mean((test_predictions_original.flatten() - y_test) ** 2)
        test_rmse = np.sqrt(test_mse)
        test_nrmse = test_rmse / np.std(y_test) if np.std(y_test) > 0 else np.inf

        return test_nrmse, best_candidate["f1_score"]
    except Exception:
        return np.inf, best_candidate["f1_score"]


all_results_by_config = {}

for config in configurations:
    T = config["T"]
    n = config["n"]
    config_key = f"T{T}_n{n}"

    Xlist = np.load(DATA_DIR / f"Xlist_T{T}_n{n}.npy", allow_pickle=True)
    ylist = np.load(DATA_DIR / f"ylist_T{T}_n{n}.npy", allow_pickle=True)

    case_results = {
        f"y{i+1}": {
            "test_nrmse": [],
            "test_f1": [],
            "perfect_match": [],
            "selection_counts": [0] * 6,
            "total_time_sec": 0.0,
            "first_run_time_sec": None,
        }
        for i in range(6)
    }

    for run_idx in range(frun):
        X = np.array(Xlist[run_idx])
        p, N, _, _ = X.shape

        split_idx = N // 2
        X_train = X[:, :split_idx, :, :]
        X_test = X[:, split_idx:, :, :]
        domain_range = [[[0, 0], [1, 1]] for _ in range(p)]

        for y_index in range(6):
            y_key = f"y{y_index+1}"
            t0 = time.perf_counter()

            y_full = np.array(ylist[run_idx][y_index])
            y_train = y_full[:split_idx]
            y_test = y_full[split_idx:]
            true_vars = ground_truth[y_index]

            best_candidate = select_best_hyperparameters(
                X_train,
                y_train,
                true_vars,
                p,
                domain_range,
                lam1_values,
                lam2_values,
                model_params,
                epsilon,
            )

            test_nrmse, test_f1 = evaluate_on_test_set(
                best_candidate,
                X_test,
                y_test,
                p,
                domain_range,
                model_params,
            )

            selected_vars = set(best_candidate["selected_vars"])
            perfect_match = 1 if selected_vars == true_vars else 0

            case_results[y_key]["test_nrmse"].append(test_nrmse)
            case_results[y_key]["test_f1"].append(test_f1)
            case_results[y_key]["perfect_match"].append(perfect_match)

            for var_idx in selected_vars:
                case_results[y_key]["selection_counts"][var_idx] += 1

            elapsed = time.perf_counter() - t0
            case_results[y_key]["total_time_sec"] += elapsed
            if run_idx == 0:
                case_results[y_key]["first_run_time_sec"] = elapsed

    all_results_by_config[config_key] = case_results

rows = []

for config_key, config_results in all_results_by_config.items():
    for y_key, metrics in config_results.items():
        test_nrmse_mean = np.mean(metrics["test_nrmse"])
        test_nrmse_std = np.std(metrics["test_nrmse"])
        test_f1_mean = np.mean(metrics["test_f1"])
        test_f1_std = np.std(metrics["test_f1"])

        row = {
            "method": "MFDNN",
            "config": config_key,
            "response": y_key,
            "nrmse_mean": test_nrmse_mean,
            "nrmse_std": test_nrmse_std,
            "f1_mean": test_f1_mean,
            "f1_std": test_f1_std,
            "perfect_match": f"{sum(metrics['perfect_match'])}/{frun}",
            "total_time_sec": round(metrics["total_time_sec"], 3),
            "first_run_time_sec": round(metrics["first_run_time_sec"], 3)
            if metrics["first_run_time_sec"] is not None
            else None,
        }

        for var_idx in range(6):
            row[f"X{var_idx + 1}"] = metrics["selection_counts"][var_idx]

        rows.append(row)

df_results = pd.DataFrame(rows)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_results.to_csv(out_path, index=False)

print(df_results.to_string(index=False))
print(f"Saved results to: {out_path}")

method   config response  nrmse_mean  nrmse_std  f1_mean   f1_std perfect_match  total_time_sec  first_run_time_sec  X1  X2  X3  X4  X5  X6
 MFDNN T16_n200       y1    0.356447   0.027787 1.000000 0.000000         50/50          94.714               1.890  50  50   0   0   0   0
 MFDNN T16_n200       y2    0.359332   0.026063 0.995000 0.035000         49/50          92.804               1.663   1  50   0   1  50  50
 MFDNN T16_n200       y3    0.367196   0.027543 0.985778 0.049443         46/50          91.867               1.827  50   4  50  50   3  50
 MFDNN T16_n200       y4    0.375542   0.027668 0.986000 0.074860         48/50          95.567               1.732  50  50   2   1   1   1
 MFDNN T16_n200       y5    0.366850   0.028686 0.990476 0.050395         48/50          93.546               1.651   1  50   2   1  50  50
 MFDNN T16_n200       y6    0.374131   0.026970 0.989143 0.043477         47/50          92.182               1.700  50   2  50  49   2  50
 MFDNN T16_n400     